In [ ]:
import pandas as pd
import os
from pathlib import Path
import re

filename = "beatles_metadata/tracks.csv"
df = pd.read_csv(filename)
df = df[['URI', 'Title', 'Year', 'Album', 'Duration', 'Tempo', 'Time_signature']]
df['title_regularized'] = df['Title'].apply(lambda x: re.sub(r" {2,}", " ", re.sub(r"[^\w\s]", "", re.sub(r"(?: - ).*", "", x))))
df.head()

,URI,Title,Year,Album,Popularity,Duration,Key,Mode,Tempo,Time_signature,...,Weeks at No1 in UK (The Guardian),Highest position (Billboard),Weeks at No1 (Billboard),Top 50 (Billboard),Top 50 (Ultimate classic rock),Top 50 (Rolling Stone),Top 50 (NME),Top 50 (Top50songs.org),"Top 50 (USA today, 2017)","Top 50 (Vulture, by Bill Wyman)"
0,spotify:track:2FDEHIMkjxFLzj688M2I3h,(You're So Square) Baby I Don't Care - Studio Jam,NaN,The Beatles,29.0,43.0,9.0,1.0,112.173,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,spotify:track:2HvTGx5fzFGpHSyRNvXd9T,12-bar Original,1965,Anthology 2,31.0,175.0,9.0,1.0,122.678,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,spotify:track:34dsKRJHIadrrNdCDtMwGn,A Beginning - Anthology 3 Version,NaN,Anthology 3,29.0,50.0,0.0,1.0,90.588,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,spotify:track:0hKRSZhUGEhKU6aNSPBACZ,A Day in the Life,1967,Sgt. Pepper's Lonely Hearts Club Band,65.0,335.0,4.0,0.0,163.219,4.0,...,NaN,NaN,NaN,NaN,1.0,1.0,2.0,18.0,1.0,1.0
4,spotify:track:5J2CHimS7dWYMImCHkEFaJ,A Hard Day's Night,1964,A Hard Day's Night,71.0,152.0,0.0,1.0,138.514,4.0,...,3.0,1.0,2.0,8.0,18.0,11.0,19.0,19.0,15.0,41.0


In [20]:
import requests

url = "https://itunes.apple.com/search"
params = {
    "term": "Youre So Square Baby I Dont Care The Beatles",
    "entity": "song",
    "attribute": "songTerm",
    "limit": 10
}

r = requests.get(url, params=params)
data = r.json()

# Inspect one result
track = data["results"][0]
print(track["trackName"])
print(track["previewUrl"])

(You're So Square) Baby I Don’t Care (Studio Jam)
https://audio-ssl.itunes.apple.com/itunes-assets/AudioPreview211/v4/eb/c3/7f/ebc37ff9-3fcb-9198-afe3-b6efdad95f2d/mzaf_248611289598540473.plus.aac.p.m4a


In [ ]:
import time
import tqdm

song_titles = list(df['title_regularized'])

url = "https://itunes.apple.com/search"

result_df = pd.DataFrame(columns=["idx", "track", "artist", "exception"])


for idx in tqdm.tqdm(range(len(song_titles))):
    # if idx == 3:
    #     break
    title = song_titles[idx]
    search_term = f"{title} The Beatles"
    params = {
        "term": search_term,
        "entity": "song",
        # "attribute": "songTerm",
        "limit": 10
    }
    try:
        r = requests.get(url, params=params)
        data = r.json()

        # Inspect one result
        track = data["results"][0]
        print(track["trackName"], track["artistName"])
        preview_url = track["previewUrl"]
        audio = requests.get(preview_url).content

        result_df.loc[len(result_df)] = [idx, track["trackName"], track["artistName"], ""]
        with open(f"beatles_audio/{idx}.m4a", "wb") as f:
            f.write(audio)
    except Exception as e:
        result_df.loc[len(result_df)] = [idx, title, "The Beatles", str(e)]
        print(f"exception at idx {idx}: {e}")
    time.sleep(1)

  0%|          | 0/285 [00:00<?, ?it/s]

(You're So Square) Baby I Don’t Care (Studio Jam) The Beatles


  0%|          | 1/285 [00:02<09:39,  2.04s/it]

12-Bar Original (Take 2 Edited) The Beatles


  1%|          | 2/285 [00:05<12:32,  2.66s/it]

...

 99%|█████████▉| 283/285 [14:03<00:06,  3.27s/it]

Young Blood (Live at the BBC for "Pop Go The Beatles" / 11th June, 1963) The Beatles


100%|█████████▉| 284/285 [14:06<00:03,  3.18s/it]

Your Mother Should Know The Beatles


100%|██████████| 285/285 [14:09<00:00,  2.98s/it]


In [ ]:
result_df.to_csv('beatles_metadata/tracks_loaded.csv')

In [49]:
import librosa 
import IPython.display as ipd

song_path = 'beatles_audio/3.m4a'
x, sr = librosa.load(song_path, sr=None, mono=True)
print('Duration: {:.2f}s, {} samples'.format(x.shape[-1] / sr, x.size))
idx += 1

ipd.Audio(data=x, rate=sr)

C:\Users\willl\AppData\Local\Temp\ipykernel_23976\1648044882.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  x, sr = librosa.load(song_path, sr=None, mono=True)


Duration: 29.93s, 1319872 samples
